# 🌐 Day 01a: IP Addressing — IPv4, Classes, CIDR & Private Networks

---

## 🎯 What You'll Master Today
- IPv4 structure (32-bit dotted decimal)
- Classful addressing (A to E)
- Private vs Public IPs
- CIDR notation / prefix length
- Special addresses

---

### 🎭 The Analogy

An IP address is like a **postal address**: network ID = city, host ID = house number.  
Classes were like pre-sized city plots (S/M/L). CIDR lets you pick the exact plot size.

```
╔══════════════════════════════════════════════════════════════════════╗
║                    IPv4 ADDRESS STRUCTURE (32 bits)                  ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║   192   .  168   .   1    .  100                                    ║
║  11000000 10101000 00000001 01100100                                ║
║  ├── 8 ──┤├── 8 ──┤├── 8 ──┤├── 8 ──┤                             ║
║                                                                      ║
║  ├──── Network ID ─────┤├── Host ID ──┤  ← /24 subnet mask        ║
║  255  .  255  .  255  .   0                                         ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================
# 1. IPv4 Address Parser
# ============================================

class IPv4Address:
    def __init__(self, ip_str):
        self.ip_str = ip_str
        self.octets = list(map(int, ip_str.split('.')))
        self.binary = self._to_binary()
        self.decimal = self._to_decimal()
    
    def _to_binary(self):
        return '.'.join(f'{o:08b}' for o in self.octets)
    
    def _to_decimal(self):
        return sum(o << (8 * (3 - i)) for i, o in enumerate(self.octets))
    
    def get_class(self):
        first = self.octets[0]
        if first < 128:   return 'A', '/8',  'Large networks (16M hosts)'
        if first < 192:   return 'B', '/16', 'Medium networks (65K hosts)'
        if first < 224:   return 'C', '/24', 'Small networks (254 hosts)'
        if first < 240:   return 'D', 'N/A', 'Multicast'
        return 'E', 'N/A', 'Reserved/Experimental'
    
    def is_private(self):
        o = self.octets
        return (o[0] == 10 or                           # 10.0.0.0/8
                (o[0] == 172 and 16 <= o[1] <= 31) or  # 172.16.0.0/12
                (o[0] == 192 and o[1] == 168))          # 192.168.0.0/16
    
    def analyze(self):
        cls, prefix, desc = self.get_class()
        print(f"  ═══ IP Address Analysis: {self.ip_str} ═══\n")
        print(f"    Dotted Decimal : {self.ip_str}")
        print(f"    Binary         : {self.binary}")
        print(f"    Decimal        : {self.decimal}")
        print(f"    Class          : {cls} (default {prefix})")
        print(f"    Description    : {desc}")
        print(f"    Private?       : {'✅ Yes' if self.is_private() else '❌ No (Public)'}")

# Test various IPs
for ip in ['10.0.1.50', '172.16.5.1', '192.168.1.100', '8.8.8.8', '224.0.0.1']:
    IPv4Address(ip).analyze()
    print()

In [ ]:
# ============================================
# 2. IP Class Reference Table
# ============================================

print("""
  ╔══════════════════════════════════════════════════════════════════════════╗
  ║                     IPv4 CLASSFUL ADDRESSING                            ║
  ╠═══════╤════════════╤═══════════╤═══════════════╤═══════════════════════╣
  ║ Class │ First Octet│  Prefix   │  # Networks   │    # Hosts/Network    ║
  ╠═══════╪════════════╪═══════════╪═══════════════╪═══════════════════════╣
  ║   A   │  1 – 126   │    /8     │     126       │   16,777,214 (16M)    ║
  ║   B   │ 128 – 191  │   /16     │   16,384      │       65,534 (65K)    ║
  ║   C   │ 192 – 223  │   /24     │ 2,097,152     │          254          ║
  ║   D   │ 224 – 239  │   N/A     │  Multicast    │          N/A          ║
  ║   E   │ 240 – 255  │   N/A     │  Reserved     │          N/A          ║
  ╚═══════╧════════════╧═══════════╧═══════════════╧═══════════════════════╝

  Private IP Ranges (RFC 1918):
    10.0.0.0    –  10.255.255.255   (10.0.0.0/8)      Class A
    172.16.0.0  –  172.31.255.255   (172.16.0.0/12)    Class B
    192.168.0.0 –  192.168.255.255  (192.168.0.0/16)   Class C

  Special Addresses:
    127.0.0.1       Loopback (localhost)
    0.0.0.0         "This" network / default route
    255.255.255.255 Broadcast
    169.254.x.x     Link-local (APIPA — no DHCP)
""")

In [ ]:
# ============================================
# 3. CIDR Notation Calculator
# ============================================

class CIDR:
    def __init__(self, cidr_notation):
        ip, prefix = cidr_notation.split('/')
        self.ip = ip
        self.prefix = int(prefix)
        self.octets = list(map(int, ip.split('.')))
    
    def subnet_mask(self):
        mask = (0xFFFFFFFF << (32 - self.prefix)) & 0xFFFFFFFF
        return '.'.join(str((mask >> (8 * i)) & 0xFF) for i in range(3, -1, -1))
    
    def network_address(self):
        ip_int = sum(o << (8 * (3 - i)) for i, o in enumerate(self.octets))
        mask = (0xFFFFFFFF << (32 - self.prefix)) & 0xFFFFFFFF
        net = ip_int & mask
        return '.'.join(str((net >> (8 * i)) & 0xFF) for i in range(3, -1, -1))
    
    def broadcast_address(self):
        ip_int = sum(o << (8 * (3 - i)) for i, o in enumerate(self.octets))
        mask = (0xFFFFFFFF << (32 - self.prefix)) & 0xFFFFFFFF
        bcast = ip_int | ~mask & 0xFFFFFFFF
        return '.'.join(str((bcast >> (8 * i)) & 0xFF) for i in range(3, -1, -1))
    
    def total_hosts(self):
        return 2 ** (32 - self.prefix)
    
    def usable_hosts(self):
        return max(self.total_hosts() - 2, 0)
    
    def analyze(self):
        print(f"  ═══ CIDR Analysis: {self.ip}/{self.prefix} ═══\n")
        print(f"    IP Address       : {self.ip}")
        print(f"    Prefix Length    : /{self.prefix}")
        print(f"    Subnet Mask      : {self.subnet_mask()}")
        print(f"    Network Address  : {self.network_address()}")
        print(f"    Broadcast Address: {self.broadcast_address()}")
        print(f"    Total Addresses  : {self.total_hosts():,}")
        print(f"    Usable Hosts     : {self.usable_hosts():,}")

for cidr in ['192.168.1.100/24', '10.0.0.1/8', '172.16.5.0/20', '192.168.1.0/30']:
    CIDR(cidr).analyze()
    print()

## 📝 Interview Questions

| # | Question | Key Point |
|---|----------|------------|
| 1 | IPv4 vs IPv6 address size? | IPv4: 32 bits (4.3B addresses). IPv6: 128 bits (3.4×10³⁸ addresses) |
| 2 | Why do we need private IPs? | IPv4 exhaustion. Private IPs reused behind NAT. RFC 1918 ranges |
| 3 | What is CIDR? | Classless Inter-Domain Routing. Variable-length prefix replaces rigid classes |
| 4 | Network vs broadcast address? | Network = all host bits 0 (identifies subnet). Broadcast = all host bits 1 |
| 5 | What is 127.0.0.1? | Loopback address. Packets never leave the host. Used for localhost testing |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • IPv4 = 32 bits, 4 octets, dotted decimal              │
## │  • Classes A-E: rigid sizes, mostly obsolete now          │
## │  • CIDR: flexible prefix length (/0 to /32)              │
## │  • Private ranges: 10.x, 172.16-31.x, 192.168.x         │
## │  • Usable hosts = 2^(32-prefix) - 2                      │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Subnetting** — VLSM, subnet calculator, practice problems